<a href="https://colab.research.google.com/github/yukien462/mysite/blob/%E6%96%99%E9%87%91%E7%A2%BA%E8%AA%8D%E7%A5%A8/%E6%96%99%E9%87%91%E7%A2%BA%E8%AA%8D%E7%A5%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## はじめに
- 初めに以下のセルを実行してください
- 必要なライブラリのインポートとデータベース（PostgreSQL）からのデータ読み込みを行います
- pandas等、利用が想定されるライブラリは以下セルでインポートしています
- その他利用したいライブラリがあれば適宜インストールしてください（"!pip install ライブラリ名"でインストールも可能）
- 処理は複数回に分けても構いません

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
!pip install chardet

### PDF変換用のツール（LibreOffice）のインストール
Google Colab (Linux環境) でExcelファイルをPDFに変換するため、ヘッドレス（画面なし）で動作する LibreOffice をインストールします。  
リブレオフィス

In [ ]:
# LibreOfficeのインストール（Colab環境にPDF変換エンジンを導入します）
!apt-get update
!apt-get install -y libreoffice

### 日本語フォントのインストール
LibreOfficeで日本語が文字化けするのを防ぐため、システムに日本語フォントをインストールします。

In [ ]:
# 日本語フォントをインストールし、キャッシュを更新します
!apt-get install -y fonts-noto-cjk fonts-ipafont-gothic fonts-ipafont-mincho
!fc-cache -fv

In [ ]:
#  文字コードを調べる
import chardet

with open('/content/drive/MyDrive/請求書発行/Data.csv','rb') as f:
  data = f.read()
  result = chardet.detect(data)
  print(result)

In [83]:
"""
データ読み込み
"""
import pandas as pd
from pathlib  import Path
def Read_CSV():

  Read_folder_path = Path('/content/drive/MyDrive/請求書発行/Data')
  csv_files = sorted(Read_folder_path.glob('*.csv'))

  # CSVファイルの最初の3行をテキストとして読み込む（エラーを避けるため）
  with open(csv_files[0], mode='r', encoding='SHIFT_JIS') as f:
      # 最初の3行をそれぞれ取得して、前後の空白や改行を削除したリストを作る
      head_lines = [f.readline().strip().strip("'\"") for _ in range(3)] # i in range(3) のiを使わないため"_"を表記
  #head_data = pd.DataFrame({"ヘッダー情報": head_lines})
  # 4行目以降（実データ）をデータフレームとして読み込む
  dfs = [pd.read_csv(csv_file, skiprows=3, encoding='SHIFT_JIS') for csv_file in csv_files]
  df = pd.concat(dfs, ignore_index=True)
  return df
  #display(df)
  #print(head_lines[0])
  #print(df['利用者'][0])

In [84]:
"""
料金確認票を作成
"""
def Create_BillState(df):
  # 研究室ごとに
  df_by_LaboCode = df.groupby('利用者の研究室コード').agg({'利用料金（小計）':'sum'}).reset_index()
  display(df_by_LaboCode)

  df_contents = df[['利用者の研究室コード','利用設備','利用者の研究室','利用者','利用料金（単価）','利用時間','利用料金（単位数）','利用料金（小計）','研究室費、私費']]
  #display(df_contents[df_contents['利用者の研究室コード'] =='BIOSF'])

  return df_contents

### PDF出力関数の実装
LibreOfficeのコマンドを使って、保存したExcelファイルをPDFに変換する関数 `Print_PDF` の実装例です。　AIさんが作成

In [ ]:
import subprocess
import os
import openpyxl
from openpyxl import utils
from copy import copy

def Print_PDF(df):
    """
    データフレームを元にExcelファイルを作成し、それをPDFに変換して保存します。
    """
    # Excelテンプレートのパス
    template_path = '/content/drive/MyDrive/請求書発行/template.xlsx'
    output_dir = '/content/drive/MyDrive/請求書発行/'

    # 研究室コードごとにデータを処理する例
    grouped = df.groupby('利用者の研究室コード')

    for lab_code, group in grouped:

        # Excelテンプレートを開く
        wb = openpyxl.load_workbook(template_path)
        ws = wb.active

        # 例: 最初の行の情報をヘッダーとして書き込む
        first_row = group.iloc[0]
        ws['B3'] = f"{first_row['利用者の研究室']} 御中"
        ws['B5'] = f"{first_row['利用設備']}"
        ws['I12'] = f"{group['利用料金（小計）'].sum()}"

        # データの件数に合わせて必要な行をあらかじめ挿入し、スタイルをコピーする設計
        labo_equip_count = len(group)
        start_row = 10

        # 複数件データがある場合、テンプレートの10行目（フォーマット定義行）を下へコピー・挿入する
        if labo_equip_count > 1:
            ws.insert_rows(start_row + 1, amount=labo_equip_count - 1)
            for i in range(1, labo_equip_count):
                # 新しく挿入した行に、元の10行目のスタイルをコピー
                CopyRow(ws, src_row=start_row, dest_row=start_row + i)

        # 先ほど紹介した「一括貼り付け」関数でデータを反映させる
        # ※ groupに必要な列情報（明細に載せたいデータ）を絞って渡します
        detail_df = group[['利用者の研究室', '利用者', '利用料金（単価）', '利用時間','利用料金（単位数）', '利用料金（小計）']]
        Paste_DataFrame_To_Excel(ws, detail_df, start_row=start_row, start_col=1)
        detail_df = group[['研究室費、私費']]
        Paste_DataFrame_To_Excel(ws, detail_df, start_row=start_row, start_col=8)

        # 一時的なExcelファイル名を作成
        temp_xlsx_name = f"請求書_{first_row['利用者の研究室']}.xlsx"
        temp_xlsx_path = os.path.join(output_dir, temp_xlsx_name)

        # Excelとして一旦保存
        wb.save(temp_xlsx_path)
        wb.close()

        print(f"Excel保存完了: {temp_xlsx_name}")

        # LibreOfficeを使ってExcelをPDFに変換 (コマンドライン実行)
        cmd = [
            "libreoffice",
            "--headless",
            "--convert-to", "pdf",
            "--outdir", output_dir,
            temp_xlsx_path
        ]

        try:
            # コマンドの実行
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            print(f"PDF変換完了: 請求書_{first_row['利用者の研究室']}.pdf")
        except subprocess.CalledProcessError as e:
            print(f"PDF変換エラー ({first_row['利用者の研究室']}): {e.stderr.decode('utf-8')}")


### CSV（DataFrame）のデータをExcelに一括貼り付けする関数の例
`openpyxl.utils.dataframe.dataframe_to_rows` を使うことで、ループ記述を最小限に抑えてデータを流し込むことができます。  
- データを一行ずつ取り出す  
for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=False), start=start_row):  
     r_idx : Excelの書き込み対象行 (10, 11, 12...)  
     row   : ['データA', 'データB', 'データC'] のような1行分のデータ  

In [ ]:
from openpyxl.utils.dataframe import dataframe_to_rows

def Paste_DataFrame_To_Excel(ws, df, start_row=10, start_col=1):
    """
    DataFrameのデータをExcelの指定した開始行・開始列から一括で貼り付けます。
    ヘッダー(列名)やインデックスを除いて、データ部分のみを貼り付ける設定にしています。
    """
    # index=False: 行インデックスを書き込まない
    # header=False: 列ヘッダーを書き込まない（テンプレートに既に項目名がある場合）
    for r_idx, row in enumerate(dataframe_to_rows(df, index=False, header=False), start=start_row):
        for c_idx, value in enumerate(row, start=start_col):
            ws.cell(row=r_idx, column=c_idx, value=value)


In [ ]:
from copy import copy

def CopyRow(ws, src_row, dest_row):
    """
    src_row から dest_row へ、値とスタイル（フォント、塗りつぶし、罫線、配置など）をコピーします。
    """
    ws.row_dimensions[dest_row].height = ws.row_dimensions[src_row].height

    # 最大列までループしてセルごとにコピー
    for col in range(1, ws.max_column + 1):
        src_cell = ws.cell(row=src_row, column=col)
        dest_cell = ws.cell(row=dest_row, column=col)

        # 値のコピー
        dest_cell.value = src_cell.value

        # スタイルのコピー (copy関数を使用することで参照ではなく複製を作成します)
        if src_cell.has_style:
            dest_cell.font = copy(src_cell.font)
            dest_cell.fill = copy(src_cell.fill)
            dest_cell.border = copy(src_cell.border)
            dest_cell.alignment = copy(src_cell.alignment)
            dest_cell.number_format = src_cell.number_format
            dest_cell.protection = copy(src_cell.protection)


In [90]:
def main():
  df = Read_CSV()           # CSV読み取り
  df = Create_BillState(df) # データ加工
  Print_PDF(df)             # PDF出力

In [92]:
if __name__=="__main__":
  main()

,利用者の研究室コード,利用料金（小計）
0,BOK,37800
1,BTM,1400
2,CML,1400
3,FSA,30100
4,GKS,122500
5,GMT,44800
6,GMY,81200
7,GTAY,4900
8,GTY,58100
9,GWT,7000


> /tmp/ipykernel_858/780978664.py(11)Create_BillState()
      8 
      9   df_contents = df[['利用者の研究室コード','利用設備','利用者の研究室','利用者','利用料金（単価）','利用時間','利用料金（単位数）','利用料金（小計）','研究室費、私費']]
     10   #display(df_contents[df_contents['利用者の研究室コード'] =='BIOSF'])
---> 11   breakpoint()
     12   return df_contents

ipdb> c
Excel保存完了: 請求書_荻野研究室.xlsx
PDF変換完了: 請求書_荻野研究室.pdf
Excel保存完了: 請求書_田畑研究室.xlsx
PDF変換完了: 請求書_田畑研究室.pdf
Excel保存完了: 請求書_マリーン・ルイス 研究室.xlsx
PDF変換完了: 請求書_マリーン・ルイス 研究室.pdf
Excel保存完了: 請求書_齊藤亜紀夫研究室.xlsx
PDF変換完了: 請求書_齊藤亜紀夫研究室.pdf
Excel保存完了: 請求書_兼橋研究室.xlsx
PDF変換完了: 請求書_兼橋研究室.pdf
Excel保存完了: 請求書_村岡研究室.xlsx

Program interrupted. (Use 'cont' to resume).
> /usr/lib/python3.13/selectors.py(398)select()
    396         ready = []
    397         try:
--> 398             fd_event_list = self._selector.poll(timeout)
    399         except InterruptedError:
    400             return ready

ipdb> q
